In [3]:
import os
import numpy as np
import pandas as pd
import random
import json
from itertools import product
from datetime import datetime
import traceback
from collections import Counter

# ======================================================================
# CONFIGURATION PARAMETERS
# ======================================================================

CONFIG = {
    'repetitions': 4,            # how many times each condition × SOA is repeated
    'num_participants': 5,       # default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    # CSV file with all the breath-hold timestamps
    'timestamps_csv': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'paths': {
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    'counterbalancing': {
        'max_sequence_length': 3,  # Avoid more than 3 of same trial type in a row
        'transition_balance': True, # Balance transitions between trial types
        'latin_square': True,       # Use latin square for participant sequence generation
        'balance_breathphase': True # Ensure even distribution across breath phases
    }
}

# Noise types for looming (these can be any labels you want)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# The boundaries of the segment we will replicate if needed
SEGMENT_START = "02:08.4"  # 13th hold ends
SEGMENT_END   = "04:56.4"  # 34th hold ends

# ======================================================================
# CLASS: PPSDesignGenerator
# ======================================================================

class PPSDesignGenerator:
    """Generates CSV design files for Peripersonal Space (PPS) experiments (no audio)."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Internal property for the timestamps
        self.timestamps_df = None
        
        # Set seed for reproducibility across participants
        # But allow for per-participant variation
        self.base_seed = 42
        
        if self.config['debug_mode']:
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {sum(self.trial_counts.values())}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """Calculate how many trials for each condition (inhale/exhale/baseline) plus catch."""
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def _expand_timestamps_if_needed(self):
        """
        Expand self.timestamps_df if we don't have enough rows to cover all trials.
        We replicate the segment from SEGMENT_START to SEGMENT_END, offsetting times
        and labeling each repeated block with repetition=1,2,3,... The original rows
        get repetition=0.
        """
        if self.timestamps_df is None or self.timestamps_df.empty:
            return
        
        # 1) Figure out how many total trials we have
        total_trials = sum(self.trial_counts.values())
        
        # 2) Check how many breath-hold rows are in the original CSV
        original_count = len(self.timestamps_df)
        if original_count >= total_trials:
            if self.config['debug_mode']:
                print(f"No expansion needed; CSV has {original_count} holds, need {total_trials}.")
            # Just set repetition=0 for all
            self.timestamps_df["repetition"] = 0
            return
        
        if self.config['debug_mode']:
            print(f"Expanding timestamps: have {original_count}, need {total_trials}.")
        
        # Sort the DataFrame by milliseconds to ensure order
        self.timestamps_df = self.timestamps_df.sort_values(by="milliseconds").reset_index(drop=True)
        
        # Identify the row indices for the segment we replicate
        seg_start_idx = self.timestamps_df.index[self.timestamps_df["timestamp"] == SEGMENT_START].tolist()
        seg_end_idx   = self.timestamps_df.index[self.timestamps_df["timestamp"] == SEGMENT_END].tolist()
        
        if not seg_start_idx or not seg_end_idx:
            raise ValueError(f"Could not find start='{SEGMENT_START}' or end='{SEGMENT_END}' in CSV.")
        
        start_i = seg_start_idx[0]
        end_i   = seg_end_idx[0]
        if end_i <= start_i:
            raise ValueError(f"Segment end index <= start index. Check your timestamps.")
        
        intro_df  = self.timestamps_df.iloc[:start_i].copy()
        repeat_df = self.timestamps_df.iloc[start_i:end_i+1].copy()
        outro_df  = self.timestamps_df.iloc[end_i+1:].copy()
        
        # Label original rows with repetition=0
        intro_df["repetition"]  = 0
        repeat_df["repetition"] = 0
        outro_df["repetition"]  = 0
        
        # Check how many we have if we keep the entire original CSV
        current_count = len(intro_df) + len(repeat_df) + len(outro_df)
        seg_len = len(repeat_df)
        
        if current_count >= total_trials:
            self.timestamps_df = pd.concat([intro_df, repeat_df, outro_df], ignore_index=True)
            return
        
        # Otherwise, replicate the repeat block until we have enough
        block_duration_ms = repeat_df["milliseconds"].iloc[-1] - repeat_df["milliseconds"].iloc[0]
        block_duration_samples = repeat_df["sample_index"].iloc[-1] - repeat_df["sample_index"].iloc[0]
        block_duration_sec = block_duration_ms / 1000.0
        
        def shift_timestamp(ts, offset_sec):
            if not isinstance(ts, str):
                return ts
            parts = ts.split(":")
            if len(parts) != 2:
                return ts
            mm = int(parts[0])
            ss = float(parts[1])
            total_s = mm * 60 + ss
            new_s = total_s + offset_sec
            new_mm = int(new_s // 60)
            new_ss = new_s % 60
            return f"{new_mm:02}:{new_ss:04.1f}"
        
        repeated_blocks = []
        offset_sec   = block_duration_sec
        offset_ms    = block_duration_ms
        offset_samps = block_duration_samples
        
        repetition_index = 1  # original = 0, so first copy = 1
        
        while current_count < total_trials:
            block_copy = repeat_df.copy()
            block_copy["timestamp"] = block_copy["timestamp"].apply(lambda t: shift_timestamp(t, offset_sec))
            block_copy["milliseconds"] = block_copy["milliseconds"] + offset_ms
            block_copy["sample_index"] = block_copy["sample_index"] + offset_samps
            block_copy["repetition"] = repetition_index
            repeated_blocks.append(block_copy)
            current_count += seg_len
            repetition_index += 1
            offset_sec   += block_duration_sec
            offset_ms    += block_duration_ms
            offset_samps += block_duration_samples
        
        self.timestamps_df = pd.concat([intro_df, repeat_df] + repeated_blocks + [outro_df], ignore_index=True)
        if self.config['debug_mode']:
            print(f"Final expanded CSV has {len(self.timestamps_df)} breath holds (needed {total_trials}).")
    
    def parse_timestamps(self):
        """Parse the box breathing timestamps CSV once, storing in self.timestamps_df, then expand if needed."""
        if self.timestamps_df is not None:
            return self.timestamps_df
        
        csv_path = self.config['timestamps_csv']
        if not os.path.exists(csv_path):
            print(f"ERROR: Timestamps CSV not found at {csv_path}")
            self.timestamps_df = pd.DataFrame()
            return self.timestamps_df
        
        try:
            df = pd.read_csv(csv_path)
            # Ensure 'breathphase' column is present
            if 'breathphase' not in df.columns:
                df['breathphase'] = ['inhale' if i % 2 == 0 else 'exhale' for i in range(len(df))]
            if 'time_seconds' not in df.columns:
                df['time_seconds'] = df['milliseconds'] / 1000.0
            df["repetition"] = 0
            self.timestamps_df = df
            
            if self.config['debug_mode']:
                print(f"Parsed {len(df)} timestamps from {csv_path}")
            
            self._expand_timestamps_if_needed()
            return self.timestamps_df
        except Exception as e:
            print(f"Error parsing timestamps CSV: {e}")
            traceback.print_exc()
            self.timestamps_df = pd.DataFrame()
            return self.timestamps_df
    
    def _generate_latin_square(self, factors, participant_id):
        """
        Generate a latin square ordering of factors for this participant.
        This ensures different participants get different factor orderings.
        """
        n = len(factors)
        base_sequence = list(range(n))
        rotation = participant_id % n
        sequence = base_sequence[rotation:] + base_sequence[:rotation]
        return [factors[i] for i in sequence]
    
    def _check_sequence_balance(self, sequence, max_length=3):
        """Check if a sequence has no more than max_length of same value in a row."""
        if not sequence:
            return True
        current_val = sequence[0]
        current_count = 1
        for val in sequence[1:]:
            if val == current_val:
                current_count += 1
                if current_count > max_length:
                    return False
            else:
                current_val = val
                current_count = 1
        return True
    
    def _check_transition_balance(self, sequence):
        """
        Check if transitions between values are balanced.
        e.g., A→B occurs similar number of times as B→A.
        """
        if len(sequence) < 3:
            return True
        transitions = {}
        for i in range(len(sequence) - 1):
            transition = (sequence[i], sequence[i+1])
            transitions[transition] = transitions.get(transition, 0) + 1
        for (a, b), count in transitions.items():
            reverse = (b, a)
            if reverse in transitions:
                if abs(transitions[reverse] - count) > 2:
                    return False
        return True
    
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a DataFrame with all trials (including baseline, catch, etc.)
        with enhanced counterbalancing of trial sequences and factor combinations.
        """
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        all_trials = []
        
        if self.config['counterbalancing'].get('latin_square', True):
            ordered_soas = self._generate_latin_square(soa_conditions, participant_id)
            ordered_stimuli = self._generate_latin_square(stimulus_types, participant_id)
        else:
            ordered_soas = soa_conditions
            ordered_stimuli = stimulus_types
        
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            combinations_per_rep = []
            for rep in range(self.config['repetitions']):
                stim_soa_combinations = list(product(ordered_stimuli, ordered_soas))
                rep_seed = self.base_seed + participant_id * 100 + rep
                random.seed(rep_seed)
                random.shuffle(stim_soa_combinations)
                combinations_per_rep.append(stim_soa_combinations)
            all_combinations = [combo for rep_combos in combinations_per_rep for combo in rep_combos]
            for stim_type, soa in all_combinations:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # baseline & main trials have tactile
                })
        
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        design_df = pd.DataFrame(all_trials)
        max_attempts = 20
        best_sequence = None
        best_score = -1
        max_sequence = self.config['counterbalancing'].get('max_sequence_length', 3)
        
        for attempt in range(max_attempts):
            attempt_seed = self.base_seed + participant_id * 1000 + attempt
            random.seed(attempt_seed)
            shuffled_df = design_df.sample(frac=1).reset_index(drop=True)
            trial_sequence = shuffled_df['trial_type'].tolist()
            sequence_ok = self._check_sequence_balance(trial_sequence, max_sequence)
            transition_ok = (not self.config['counterbalancing'].get('transition_balance', True) or 
                             self._check_transition_balance(trial_sequence))
            score = 0
            if sequence_ok:
                score += 1
            if transition_ok:
                score += 1
            transition_counts = {}
            for i in range(len(trial_sequence) - 1):
                transition = (trial_sequence[i], trial_sequence[i+1])
                transition_counts[transition] = transition_counts.get(transition, 0) + 1
            values = list(transition_counts.values())
            entropy = -sum([(v/len(values)) * np.log(v/len(values)) for v in values])
            score += entropy
            if score > best_score:
                best_score = score
                best_sequence = shuffled_df
            if sequence_ok and transition_ok and entropy > 1.5:
                break
        
        if best_sequence is not None:
            design_df = best_sequence
        
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            trial_counts = design_df['trial_type'].value_counts()
            print(f"\nTrial type distribution:")
            print(trial_counts)
            trial_sequence = design_df['trial_type'].tolist()
            max_run = 1
            current_run = 1
            current_type = trial_sequence[0]
            for t in trial_sequence[1:]:
                if t == current_type:
                    current_run += 1
                    max_run = max(max_run, current_run)
                else:
                    current_type = t
                    current_run = 1
            print(f"Maximum consecutive trials of same type: {max_run}")
        
        return design_df
    
    def assign_breath_holds(self, design_df):
        """
        Assign each trial to an inhale or exhale row in self.timestamps_df.
        Now with enhanced counterbalancing of breath phases across conditions.
        """
        timestamps_df = self.parse_timestamps()  # ensures expansion is done
        if timestamps_df.empty:
            design_df['breathphase'] = None
            design_df['milliseconds'] = None
            design_df['jittered_ms'] = None
            design_df['soa_ms'] = None
            design_df['timestamp_original'] = None
            design_df['timestamp_after_jitter'] = None
            design_df['timestamp_with_soa'] = None
            return design_df
        
        skip_count = int(len(timestamps_df) * 0.1)
        usable = timestamps_df.iloc[skip_count:].copy()
        inhale_rows = usable[usable['breathphase'] == 'inhale']
        exhale_rows = usable[usable['breathphase'] == 'exhale']
        inhale_pool = inhale_rows.index.tolist()
        exhale_pool = exhale_rows.index.tolist()
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        phase_counters = {}
        trial_assignments = {}
        
        for trial_type in design_df['trial_type'].unique():
            type_indices = design_df[design_df['trial_type'] == trial_type].index.tolist()
            if (trial_type, 'inhale') not in phase_counters:
                phase_counters[(trial_type, 'inhale')] = 0
            if (trial_type, 'exhale') not in phase_counters:
                phase_counters[(trial_type, 'exhale')] = 0
            for idx in type_indices:
                inhale_count = phase_counters[(trial_type, 'inhale')]
                exhale_count = phase_counters[(trial_type, 'exhale')]
                if trial_type == 'inhalation' and inhale_pool:
                    use_inhale = True
                elif trial_type == 'exhalation' and exhale_pool:
                    use_inhale = False
                else:
                    if inhale_count < exhale_count and inhale_pool:
                        use_inhale = True
                    elif exhale_count < inhale_count and exhale_pool:
                        use_inhale = False
                    else:
                        if not inhale_pool:
                            use_inhale = False
                        elif not exhale_pool:
                            use_inhale = True
                        else:
                            use_inhale = random.choice([True, False])
                if use_inhale and inhale_pool:
                    timestamp_idx = inhale_pool.pop(0)
                    row_ts = timestamps_df.loc[timestamp_idx]
                    trial_assignments[idx] = row_ts
                    phase_counters[(trial_type, 'inhale')] += 1
                elif exhale_pool:
                    timestamp_idx = exhale_pool.pop(0)
                    row_ts = timestamps_df.loc[timestamp_idx]
                    trial_assignments[idx] = row_ts
                    phase_counters[(trial_type, 'exhale')] += 1
                else:
                    raise ValueError(f"Ran out of timestamp rows for trial assignment. Need more timestamps!")
        
        design_df['breathphase'] = design_df.index.map(
            lambda i: trial_assignments[i]['breathphase'] if i in trial_assignments else None
        )
        design_df['milliseconds'] = design_df.index.map(
            lambda i: trial_assignments[i]['milliseconds'] if i in trial_assignments else None
        )
        design_df['jittered_ms'] = design_df.apply(
            lambda row: row['milliseconds'] + row['jitter_ms'] if pd.notnull(row['milliseconds']) else None,
            axis=1
        )
        design_df['soa_ms'] = design_df.apply(
            lambda row: row['jittered_ms'] + row['soa_value_ms']
            if (pd.notnull(row['jittered_ms']) and row['trial_type'] != 'catch') else None,
            axis=1
        )
        def ms_to_timestamp(ms):
            if pd.isnull(ms):
                return None
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_df['timestamp_original'] = design_df['milliseconds'].apply(ms_to_timestamp)
        design_df['timestamp_after_jitter'] = design_df['jittered_ms'].apply(ms_to_timestamp)
        design_df['timestamp_with_soa'] = design_df['soa_ms'].apply(ms_to_timestamp)
        
        if self.config['debug_mode']:
            print("\nBreath phase allocation by trial type:")
            for trial_type in design_df['trial_type'].unique():
                type_df = design_df[design_df['trial_type'] == trial_type]
                phase_counts = type_df['breathphase'].value_counts()
                print(f"  {trial_type}: {phase_counts.to_dict()}")
        
        return design_df
    
    def finalize_design_csv(self, design_df):
        """
        1) Rename columns to final naming:
           - stimulus_type -> looming_stimulus_type
           - timestamp_original -> retentionphase_timestamp
           - timestamp_after_jitter -> looming_stimulus_timestamp
           - timestamp_with_soa -> tactile_stimulus_timestamp
        2) For baseline trials: set looming_stimulus_type='none', looming_stimulus_timestamp=NaN
        3) For catch trials: set tactile_stimulus_timestamp=NaN
        4) Return the final DataFrame (ready to save).
        """
        design_df.rename(columns={'stimulus_type': 'looming_stimulus_type'}, inplace=True)
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'looming_stimulus_type'] = 'none'
        design_df.rename(columns={
            'timestamp_original':      'retentionphase_timestamp',
            'timestamp_after_jitter':  'looming_stimulus_timestamp',
            'timestamp_with_soa':      'tactile_stimulus_timestamp'
        }, inplace=True)
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = np.nan
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = np.nan
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        final_cols = [
            'participant_id',
            'trial_number',
            'trial_type',
            'breathphase',
            'retentionphase_timestamp',
            'looming_stimulus_timestamp',
            'tactile_stimulus_timestamp',
            'looming_stimulus_type',
            'soa_value_ms',
            'jitter_ms',
            'is_tactile',
            'expected_response'
        ]
        final_cols = [c for c in final_cols if c in design_df.columns]
        design_df = design_df[final_cols]
        self._validate_design(design_df)
        return design_df
    
    def _validate_design(self, design_df):
        """Check if the final design meets our counterbalancing criteria."""
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        trial_counts = design_df['trial_type'].value_counts()
        print(trial_counts)
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            type_df = design_df[design_df['trial_type'] == trial_type]
            stim_counts = type_df['looming_stimulus_type'].value_counts()
            print(f"  {trial_type}: {stim_counts.to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            type_df = design_df[design_df['trial_type'] == trial_type]
            soa_counts = type_df['soa_value_ms'].value_counts()
            print(f"  {trial_type}: {soa_counts.to_dict()}")
        print("\nBreath phase distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            type_df = design_df[design_df['trial_type'] == trial_type]
            phase_counts = type_df['breathphase'].value_counts()
            print(f"  {trial_type}: {phase_counts.to_dict()}")
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        max_consecutive = 1
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
                max_consecutive = max(max_consecutive, current_count)
            else:
                length_key = f"{current_type}_{current_count}"
                consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
                current_type = t
                current_count = 1
        length_key = f"{current_type}_{current_count}"
        consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
        print(f"\nMaximum consecutive trials of same type: {max_consecutive}")
        print("Distribution of consecutive runs:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
        transitions = {}
        for i in range(len(trial_sequence) - 1):
            transition = (trial_sequence[i], trial_sequence[i+1])
            transitions[transition] = transitions.get(transition, 0) + 1
        print("\nTransition counts between trial types:")
        for transition, count in sorted(transitions.items()):
            print(f"  {transition[0]} → {transition[1]}: {count}")
    
    def process_participant(self, participant_id):
        """
        Generate the design, assign breath holds, finalize columns,
        and save as a CSV for one participant.
        """
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None, num_participants=None):
        """
        Generate the design CSVs for multiple participants.
        """
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Counterbalancing settings: {self.config['counterbalancing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {
                'success': success,
                'csv_path': path
            }
        return results

# ======================================================================
# MAIN
# ======================================================================

def main():
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()  # by default, uses num_participants=5
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")

if __name__ == "__main__":
    main()


Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Counterbalancing settings: {'max_sequence_length': 3, 'transition_balance': True, 'latin_square': True, 'balance_breathphase': True}

=== Processing participant 0 ===

Trial type distribution:
trial_type
inhalation    80
exhalation    80
baseline      80
catch         24
Name: count, dtype: int64
Maximum consecutive trials of same type: 80
Parsed 38 timestamps from C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv
Expand

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_18820\2397889412.py", line 607, in process_participant
    design_df = self.assign_breath_holds(design_df)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_18820\2397889412.py", line 464, in assign_breath_holds
    raise ValueError(f"Ran out of timestamp rows for trial assignment. Need more timestamps!")
ValueError: Ran out of timestamp rows for trial assignment. Need more timestamps!
Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_18820\2397889412.py", line 607, in process_participant
    design_df = self.assign_breath_holds(design_df)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_18820\2397889412.py", line 464, in assign_breath_holds
    raise ValueError(f"Ran out of timestamp rows for trial assignment. Need more timestam